Model pracuje s daty ve složce X, TimeSeriesDataset projde tuto složku rekurzivně a načte všechny npy soubory,
tyto soubory musí být tvaru 3 časových řad o 4096 číslech.

In [ ]:
import torch
import torch.nn as nn

class ConcatBlockConv5(nn.Module):
    def __init__(self, in_ch, out_ch, k=32, act=nn.SiLU):
        super().__init__()
        
        def make_block(kernel_size):
            return nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding='same'),
                nn.BatchNorm1d(out_ch),
                act()
            )

        self.c1 = make_block(k)          # k=32
        self.c2 = make_block(k * 2)      # k=64
        self.c3 = make_block(k // 2)     # k=16
        self.c4 = make_block(k // 4)     # k=8
        self.c5 = make_block(k * 4)      # k=128
        
        c6_in_channels = (out_ch * 5) + in_ch
        
        self.c6 = nn.Sequential(
            nn.Conv1d(c6_in_channels, out_ch, kernel_size=1),
            nn.BatchNorm1d(out_ch),
            act()
        )

    def forward(self, x):
        x1 = self.c1(x)
        x2 = self.c2(x)
        x3 = self.c3(x)
        x4 = self.c4(x)
        x5 = self.c5(x)

        out = torch.cat([x1, x2, x3, x4, x5, x], dim=1)
        
        return self.c6(out)

In [ ]:
class GWNet(nn.Module):
    def __init__(self, channels=3, classes=1):
        super().__init__()

        self.b1 = ConcatBlockConv5(channels, 32, k=32)
        self.p1 = nn.MaxPool1d(2)

        self.b2 = ConcatBlockConv5(32, 64, k=32)
        self.p2 = nn.MaxPool1d(2)

        self.b3 = ConcatBlockConv5(64, 128, k=32)
        self.p3 = nn.MaxPool1d(2)

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128, classes)

    def forward(self, x):
        x = self.p1(self.b1(x))
        x = self.p2(self.b2(x))
        x = self.p3(self.b3(x))
        x = self.gap(x).squeeze(-1)
        return self.fc(x)

In [ ]:
from torch.utils.data import Dataset
import torch
import numpy as np
import pandas as pd
import os
import glob

class TimeSeriesDataset(Dataset):
    def __init__(self, root_dir, labels_csv, return_raw_only, return_process_only):
        self.root_dir = root_dir
        self.return_raw_only = return_raw_only
        self.return_process_only=return_process_only
        df = pd.read_csv(labels_csv)
        self.label_map = {str(name).replace('.npy', ''): int(label)
                          for name, label in zip(df.iloc[:, 0], df.iloc[:, 1])}

        all_files = glob.glob(os.path.join(root_dir, "**", "*.npy"), recursive=True)

        self.files = [f for f in all_files
                      if os.path.splitext(os.path.basename(f))[0] in self.label_map]
        self.process_number = 10
    
    def __len__(self):
        return len(self.files)


    def __getitem__(self, idx):
        file = self.files[idx]
        data = read_file(file)
        d1, d2, d3 = data
    
        if self.return_raw_only:
            stacked = np.stack([d1, d2, d3], axis=0)
        elif self.return_process_only:
            p1, p2, p3 = preprocess(data, self.process_number)
            stacked = np.stack([p1, p2, p3], axis=0)
        else:
            p1, p2, p3 = preprocess(d1, d2, d3)
            stacked = np.stack([d1, p1, d2, p2, d3, p3], axis=0)
    
        X = torch.tensor(stacked, dtype=torch.float32)
        base = os.path.splitext(os.path.basename(file))[0]
        y = torch.tensor([self.label_map[base]], dtype=torch.float32)
        return X, y#, base


In [ ]:
from torch.utils.data import DataLoader, random_split
n_total = len(dataset)
n_val = int(0.1 * n_total)
n_train = n_total - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=8, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4)

In [ ]:
def read_file(fname):
    data = np.load(fname)
    return data

from scipy import signal
from scipy.signal.windows import tukey
window = tukey(4096, 0.1)
sos = signal.butter(8, [20, 500], btype="bandpass", output="sos", fs=2048)
def bandpass(x):
    x *= window
    for i in range(3):
        x[i] = signal.sosfilt(sos, x[i])
    return x

scaling = [1.5e-20, 1.5e-20, 0.5e-20]
def preprocess(data):
    p1, p2, p3 = bandpass(data)
    p1 = p1 / scaling[0]
    p2 = p2 / scaling[1]
    p3 = p3 / scaling[2]
    return p1, p2, p3

def load_and_preprocess(file_path):
    data = read_file(file_path)
    p1, p2, p3 = preprocess(data, 0)
    stacked = np.stack([p1, p2, p3], axis=0)
    return stacked

def get_model_probability(processed_data):
    input_tensor = torch.tensor(processed_data).float()
    input_tensor = input_tensor.unsqueeze(0)
    device = next(model.parameters()).device
    input_tensor = input_tensor.to(device)
    with torch.no_grad():
        outputs = model(input_tensor)
        prob = torch.sigmoid(outputs).item()
    return prob

In [ ]:
def augment(x):
    # x: (3, T)

    # channel shuffle
    if torch.rand(1) < 0.3:
        idx = torch.randperm(3)
        x = x[idx]

    # small circular shift
    if torch.rand(1) < 0.5:
        shift = torch.randint(-20, 20, (1,))
        x = torch.roll(x, shifts=shift.item(), dims=-1)

    # time masking
    if torch.rand(1) < 0.5:
        l = x.shape[-1]
        t0 = torch.randint(0, l-50, (1,))
        t1 = t0 + torch.randint(20, 100, (1,))
        x[:, t0:t1] = 0

    return x

In [ ]:
dataset = TimeSeriesDataset(root_dir="g2net-gravitational-wave-detection/train/", 
                            labels_csv="g2net-gravitational-wave-detection/training_labels.csv",
                            return_raw_only=False, return_process_only=True)

In [ ]:
device = "cuda"

model = GWNet().to(device)
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    nesterov=True,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=40,
    eta_min=1e-4
)

EPOCHS = 40

In [ ]:

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        X = torch.stack([augment(x) for x in X])

        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss:.4f}")
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")
    if (epoch + 1) % 10 == 0:
        tag = f"epoch_{epoch+1:04d}"

        torch.save(
            model.state_dict(),
            f"residual_model_least_process{tag}.pth"
        )

        torch.save(
            model,
            f"residual_model_full_least_process{tag}.pth"
        )

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': total_loss,
        }, f"residual_checkpoint_least_process{tag}.pth")

In [ ]:
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for X, y in val_loader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)

        probs = torch.sigmoid(outputs).squeeze()
        preds = (probs > 0.5).long()

        all_labels.extend(y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)

print(f"Validation Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")